In [7]:
# %% Cell 1 — imports and base params
import numpy as np
import numba
import matplotlib.pyplot as plt
from tqdm import tqdm
import time
import os

J = 1
Tc = 2 / np.log(1 + np.sqrt(2))  # ≈ 2.269

# %% Cell 2 — Wolff step and Monte Carlo class
@numba.njit
def _wolff_step_numba(spins, wolff_marker, L, wolff_prob):
    wolff_marker.fill(0)

    i0 = np.random.randint(0, L)
    j0 = np.random.randint(0, L)
    ground_spin = spins[i0, j0]
    wolff_marker[i0, j0] = 1

    stack = np.zeros((L * L, 2), dtype=np.int64)
    stack[0, 0] = i0
    stack[0, 1] = j0
    stack_size = 1

    while stack_size > 0:
        stack_size -= 1
        ci = stack[stack_size, 0]
        cj = stack[stack_size, 1]
        for d in range(4):
            ni = (ci + (d == 1) - (d == 0)) % L
            nj = (cj + (d == 3) - (d == 2)) % L
            if wolff_marker[ni, nj] == 0 and spins[ni, nj] == ground_spin:
                if np.random.random() < wolff_prob:
                    wolff_marker[ni, nj] = 1
                    stack[stack_size, 0] = ni
                    stack[stack_size, 1] = nj
                    stack_size += 1

    N_cluster = 0
    for i in range(L):
        for j in range(L):
            if wolff_marker[i, j] == 1:
                spins[i, j] *= -1
                N_cluster += 1

    return spins, N_cluster, ground_spin


class IsingMC_Wolff:
    def __init__(self, length, temperature=0.):
        self.spins = np.random.choice([-1, 1], size=(length, length)).astype(np.int64)
        self.L = length
        self.T = temperature
        self.M = int(np.sum(self.spins))
        self.wolff_prob = None
        self.wolff_marker = np.zeros((length, length), dtype=np.int64)
        self.update_probabilities()

    def update_probabilities(self):
        if self.T != 0.:
            self.wolff_prob = 1. - np.exp(-2. / self.T)
        else:
            self.wolff_prob = 1.

    def set_temperature(self, temperature):
        self.T = temperature
        self.update_probabilities()

    def wolff_step(self):
        self.spins, N_cluster, ground_spin = _wolff_step_numba(
            self.spins, self.wolff_marker, self.L, self.wolff_prob)
        self.M -= 2 * ground_spin * N_cluster


# %% Cell 3 — parameters
L_values_data = [10, 20, 30, 40, 60, 70, 80, 90, 100]
N_per_temps   = 500
Ntherm        = 10000
Ndecorr       = 200

output_dir = "data_newpoints"
os.makedirs(output_dir, exist_ok=True)


Temp = np.array([
    1.0000000000000000, 1.0634592657106510, 1.1269185314213019, 1.1903777971319529,
    1.2538370628426039, 1.3172963285532548, 1.3807555942639058, 1.4442148599745568,
    1.5076741256852078, 1.5711333913958587, 1.6345926571065097, 1.6980519228171607,
    1.7615111885278116, 1.8249704542384626, 1.8884297199491136, 1.9518889856597645,
    2.0153482513704155, 2.0788075170810667, 2.1422667827917179, 2.2057260485023691,
    2.2691853142130203, 2.3326445799236715, 2.3961038456343227, 2.4595631113449739,
    2.5230223770556250, 2.5864816427662762, 2.6499409084769274, 2.7134001741875786,
    2.7768594398982298, 2.8403187056088810, 2.9037779713195322, 2.9672372370301834,
    3.0306965027408346, 3.0941557684514858, 3.1576150341621370, 3.2210742998727881,
    3.2845335655834393, 3.3479928312940905, 3.4114520970047417, 3.4749113627153929,
    3.5383706284260401
])

# %% Cell 4 — generate and save
def generate_for_L(L, temperatures, N_per_temps, Ntherm, Ndecorr):
    N = L * L
    n_temps = len(temperatures)
    temperatures_arr = np.zeros(n_temps * N_per_temps, dtype=np.float32)
    spins_arr        = np.zeros((n_temps * N_per_temps, N), dtype=np.int8)

    model = IsingMC_Wolff(L)

    for t_idx, T in enumerate(tqdm(temperatures, desc=f"L={L}", leave=False)):
        # fresh hot start at each temperature
        model.spins = np.random.choice([-1, 1], size=(L, L)).astype(np.int64)
        model.M = int(np.sum(model.spins))
        model.set_temperature(T)

        for _ in range(Ntherm):
            model.wolff_step()

        for row_idx in range(N_per_temps):
            for _ in range(Ndecorr):
                model.wolff_step()
            idx = t_idx * N_per_temps + row_idx
            temperatures_arr[idx] = T
            spins_arr[idx] = model.spins.flatten()

    return temperatures_arr, spins_arr


for L in L_values_data:
    t_start = time.perf_counter()
    temperatures_arr, spins_arr = generate_for_L(L, Temp, N_per_temps, Ntherm, Ndecorr)
    filename = os.path.join(output_dir, f"L{L}_ising_pointsad.npz")
    np.savez_compressed(filename, temperatures=temperatures_arr, spins=spins_arr)
    print(f"Saved L={L} to {filename}   ({time.perf_counter()-t_start:.1f}s)")

Saved L=10 to data_newpoints/L10_ising_pointsad.npz   (13.0s)


Saved L=20 to data_newpoints/L20_ising_pointsad.npz   (35.8s)


Saved L=30 to data_newpoints/L30_ising_pointsad.npz   (71.2s)


Saved L=40 to data_newpoints/L40_ising_pointsad.npz   (121.1s)


Saved L=60 to data_newpoints/L60_ising_pointsad.npz   (261.9s)


Saved L=70 to data_newpoints/L70_ising_pointsad.npz   (391.7s)


Saved L=80 to data_newpoints/L80_ising_pointsad.npz   (495.5s)


Saved L=90 to data_newpoints/L90_ising_pointsad.npz   (590.3s)


Saved L=100 to data_newpoints/L100_ising_pointsad.npz   (766.4s)
